File Search

파일 객체 생성 -> 벡터 스토어에 추가
벡터 스토어 생성

In [1]:
from dotenv import load_dotenv
import os

load_dotenv()

# 사용할 키 가져오기
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

# Open AI API 연결
from openai import OpenAI

# 연결된 open ai 객체 생성
client = OpenAI(api_key=OPENAI_API_KEY)

In [2]:
# 함수 : Open AI API 사용하는 파일 객체 생성 방법
# Client.files.create
def create_file(client, file_path):
    with open(file_path, "rb") as file_content:
        result = client.files.create(
            file = file_content,
            purpose="assistants"
        )
    print(result.id)
    return result.id            # id로 벡터 스토어에 등록

In [3]:
# file 객체 생성 : create_file => id
file_id = create_file(client, "./howto-sockets.pdf")

file-Nw9MjGbEJRnzL9e4gk8qiX


In [4]:
# 벡터 스토어 생성
vector_store = client.vector_stores.create(
    name="rlarbfbd"
)

In [7]:
vector_store

VectorStore(id='vs_6a792c6b911c819195a212b129941e92', created_at=1786326123, file_counts=FileCounts(cancelled=0, completed=0, failed=0, in_progress=0, total=0), last_active_at=1786326123, metadata={}, name='rlarbfbd', object='vector_store', status='completed', usage_bytes=0, expires_after=None, expires_at=None, description=None)

In [ ]:
# 벡터 스토어 : 파일 업로드 (저장, 추가) => 어떤 벡터 스토어에 어떤 파일을 올릴 것인지 정의
# 여러 개 생성 가능
client.vector_stores.files.create(
    vector_store_id=vector_store.id,
    file_id=file_id
)

VectorStoreFile(id='file-Nw9MjGbEJRnzL9e4gk8qiX', created_at=1786326360, last_error=None, object='vector_store.file', status='in_progress', usage_bytes=0, vector_store_id='vs_6a792c6b911c819195a212b129941e92', attributes={}, chunking_strategy=StaticFileChunkingStrategyObject(static=StaticFileChunkingStrategy(chunk_overlap_tokens=400, max_chunk_size_tokens=800), type='static'))

In [10]:
file_list = client.vector_stores.files.list(
    vector_store_id=vector_store.id
)

In [12]:
file_list

SyncCursorPage[VectorStoreFile](data=[VectorStoreFile(id='file-Nw9MjGbEJRnzL9e4gk8qiX', created_at=1786326360, last_error=None, object='vector_store.file', status='completed', usage_bytes=40838, vector_store_id='vs_6a792c6b911c819195a212b129941e92', attributes={}, chunking_strategy=StaticFileChunkingStrategyObject(static=StaticFileChunkingStrategy(chunk_overlap_tokens=400, max_chunk_size_tokens=800), type='static'))], has_more=False, object='list', first_id='file-Nw9MjGbEJRnzL9e4gk8qiX', last_id='file-Nw9MjGbEJRnzL9e4gk8qiX')

여기까지가 File Search 기반

In [13]:
input = "파이썬 코드로 소켓을 만드는 방법을 간단히 설명해줘"

response = client.responses.create(
    model="gpt-5.5",
    input=input,
    tools=[
        {
            "type": "file_search",
            "vector_store_ids": [vector_store.id]
        }
    ]
)

In [16]:
print(response.output_text)

파이썬에서는 내장 모듈인 `socket`을 사용해서 소켓을 만들 수 있습니다.

## 1. 기본 소켓 생성

```python
import socket

s = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
```

여기서:

- `AF_INET` : IPv4 주소 체계 사용
- `SOCK_STREAM` : TCP 소켓 사용

즉, 위 코드는 **IPv4 기반 TCP 소켓**을 만드는 코드입니다.

---

## 2. TCP 서버 예제

```python
import socket

server_socket = socket.socket(socket.AF_INET, socket.SOCK_STREAM)

server_socket.bind(("127.0.0.1", 5000))
server_socket.listen()

print("서버 대기 중...")

client_socket, addr = server_socket.accept()
print("연결됨:", addr)

data = client_socket.recv(1024)
print("받은 데이터:", data.decode())

client_socket.send("안녕하세요, 클라이언트!".encode())

client_socket.close()
server_socket.close()
```

서버는 다음 순서로 동작합니다.

1. 소켓 생성
2. IP와 포트에 바인딩
3. 연결 대기
4. 클라이언트 연결 수락
5. 데이터 송수신
6. 소켓 닫기

---

## 3. TCP 클라이언트 예제

```python
import socket

client_socket = socket.socket(socket.AF_INET, socket.SOCK_STREAM)

client_socket.connect(("127.0.0.1", 5000))

client_socket.send("안녕하세요, 서버!".encode())

data = client_socket.recv(1024)
pr